# Gold Company Activity Mart

**Purpose**: Company-level hiring patterns and activity metrics by sector.

**Target Table**: `workspace.gold.gold_company_activity`

**Grain**: One row per date per company per sector

**Sector Support**: Multi-sector enabled via sector_sk dimension

**Key Metrics**:
- Active jobs per company
- New postings and closures
- Hiring velocity
- Role diversity
- Geographic reach

**Data Sources**:
- `workspace.warehouse.fact_job_postings`
- `workspace.warehouse.dim_company`
- `workspace.warehouse.dim_sector`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_company_activity
USING DELTA
COMMENT 'Company-level hiring patterns across all sectors'
AS

WITH base_metrics AS (
  SELECT
    f.posting_date_sk AS activity_date_sk,
    f.sector_sk,
    f.company_sk,
    
    -- MEASURES: Job counts
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS active_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) AS new_jobs,
    COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS closed_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) - 
      COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS net_change,
    
    -- MEASURES: Diversity metrics
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.role_sk END) AS unique_roles,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.location_sk END) AS unique_locations,
    
    -- Total events
    COUNT(*) AS total_events
    
  FROM workspace.warehouse.fact_job_postings f
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
    AND f.sector_sk IS NOT NULL  -- Ensure valid sector assignment
  GROUP BY f.posting_date_sk, f.sector_sk, f.company_sk
),

temporal_enriched AS (
  SELECT
    bm.*,
    
    -- TEMPORAL METRICS: Prior period comparisons
    LAG(bm.active_jobs, 7) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk
      ORDER BY bm.activity_date_sk
    ) AS prev_week_active_jobs,
    
    LAG(bm.active_jobs, 30) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk
      ORDER BY bm.activity_date_sk
    ) AS prev_month_active_jobs,
    
    -- Week-to-date cumulative new jobs
    SUM(bm.new_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk,
        YEAR(TO_DATE(CAST(bm.activity_date_sk AS STRING), 'yyyyMMdd')),
        WEEKOFYEAR(TO_DATE(CAST(bm.activity_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY bm.activity_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS wtd_new_jobs,
    
    -- Month-to-date cumulative new jobs
    SUM(bm.new_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk,
        YEAR(TO_DATE(CAST(bm.activity_date_sk AS STRING), 'yyyyMMdd')),
        MONTH(TO_DATE(CAST(bm.activity_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY bm.activity_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS mtd_new_jobs,
    
    -- 7-day rolling average
    AVG(bm.new_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk
      ORDER BY bm.activity_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_new_jobs,
    
    -- 30-day rolling average
    AVG(bm.active_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk
      ORDER BY bm.activity_date_sk
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30day_avg_active_jobs,
    
    -- Rank companies by active jobs within each day and sector
    DENSE_RANK() OVER (
      PARTITION BY bm.activity_date_sk, bm.sector_sk
      ORDER BY CASE WHEN bm.active_jobs > 0 THEN bm.active_jobs ELSE 0 END DESC
    ) AS daily_company_rank
    
  FROM base_metrics bm
)

SELECT
  -- DIMENSIONS (sector_sk first for partitioning)
  te.sector_sk,
  te.activity_date_sk,
  te.company_sk,
  te.daily_company_rank,
  
  -- MEASURES
  te.active_jobs,
  te.new_jobs,
  te.closed_jobs,
  te.net_change,
  te.unique_roles,
  te.unique_locations,
  te.total_events,
  
  -- TEMPORAL METRICS
  te.wtd_new_jobs,
  te.mtd_new_jobs,
  CAST(te.rolling_7day_avg_new_jobs AS DECIMAL(10,2)) AS rolling_7day_avg_new_jobs,
  CAST(te.rolling_30day_avg_active_jobs AS DECIMAL(10,2)) AS rolling_30day_avg_active_jobs,
  
  -- DERIVED: Percent changes
  CASE 
    WHEN te.prev_week_active_jobs > 0 
    THEN CAST((te.active_jobs - te.prev_week_active_jobs) AS DECIMAL(10,4)) / te.prev_week_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_week,
  
  CASE 
    WHEN te.prev_month_active_jobs > 0 
    THEN CAST((te.active_jobs - te.prev_month_active_jobs) AS DECIMAL(10,4)) / te.prev_month_active_jobs
    ELSE NULL 
  END AS pct_change_vs_prev_month,
  
  -- DERIVED: Hiring velocity
  CASE 
    WHEN te.active_jobs > 0 
    THEN CAST(te.new_jobs AS DECIMAL(10,4)) / te.active_jobs
    ELSE NULL 
  END AS hiring_velocity,
  
  -- DERIVED: Job churn rate
  CASE 
    WHEN te.prev_week_active_jobs > 0 
    THEN CAST(te.closed_jobs AS DECIMAL(10,4)) / te.prev_week_active_jobs
    ELSE NULL 
  END AS job_churn_rate,
  
  -- DERIVED: Role diversity score (roles per location)
  CASE 
    WHEN te.unique_locations > 0 
    THEN CAST(te.unique_roles AS DECIMAL(10,2)) / te.unique_locations
    ELSE te.unique_roles
  END AS role_diversity_score,
  
  -- DERIVED: Geographic reach indicator
  CASE 
    WHEN te.unique_locations >= 10 THEN 'National'
    WHEN te.unique_locations >= 5 THEN 'Regional'
    WHEN te.unique_locations >= 2 THEN 'Multi-City'
    ELSE 'Local'
  END AS geographic_reach,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
WHERE te.daily_company_rank <= 200  -- Keep top 200 companies per day per sector for performance
ORDER BY te.sector_sk, te.activity_date_sk DESC, te.daily_company_rank;


In [0]:
%sql
-- Validation: Top employers by sector
SELECT
  s.sector_name,
  c.canonical_company_name,
  c.industry,
  SUM(gca.active_jobs) AS total_active_jobs,
  SUM(gca.new_jobs) AS total_new_jobs,
  ROUND(AVG(gca.hiring_velocity), 4) AS avg_hiring_velocity,
  COUNT(DISTINCT gca.activity_date_sk) AS days_active
FROM workspace.gold.gold_company_activity gca
INNER JOIN workspace.warehouse.dim_company c ON gca.company_sk = c.company_sk
INNER JOIN workspace.warehouse.dim_sector s ON gca.sector_sk = s.sector_sk
WHERE gca.activity_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 30), 'yyyyMMdd') AS INT)
GROUP BY s.sector_name, c.canonical_company_name, c.industry
ORDER BY s.sector_name, total_active_jobs DESC
LIMIT 50;
